In [2]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import os
import seaborn as sns
import gc
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, confusion_matrix, classification_report
import time
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)
from sklearn.linear_model import LogisticRegression
import xgboost as xgb
import lightgbm as lgb
from sklearn.neural_network import MLPClassifier
from mlxtend.classifier import StackingClassifier
import gdown

In [ ]:
# load data
url_train_transaction = '1-LA_hivwi-LTmk3Il6zWc2SlWkw63lIp'
url = f'https://drive.google.com/uc?id={url_train_transaction}'
output = 'train_transaction.csv'
gdown.download(url, output, quiet=False)
train_transaction = pd.read_csv(output)

url_train_identity = '1-3IcnQe1Vk6USpMFiZcc4plMM27u2BRM'
url = f'https://drive.google.com/uc?id={url_train_identity}'
output = 'train_identity.csv'
gdown.download(url, output, quiet=False)
train_identity = pd.read_csv(output)

url_test_transaction = '1--SQCKBo6r7JKFDJU1WLVSVzj7Wmblz1'
url = f'https://drive.google.com/uc?id={url_test_transaction}'
output = 'test_transaction.csv'
gdown.download(url, output, quiet=False)
test_transaction = pd.read_csv(output)

url_test_identity = '1htmHneOewTUARR48VB5rF0k3tZIGpFcj'
url = f'https://drive.google.com/uc?id={url_test_identity}'
output = 'test_identity.csv'
gdown.download(url, output, quiet=False)
test_identity = pd.read_csv(output)

# merge data
df = pd.merge(train_transaction, train_identity, on='TransactionID', how='left')
df_test = pd.merge(test_transaction, test_identity, on='TransactionID', how='left')

print("data train: ", df.shape)
print("data test: ", df_test.shape)

#ubah nama kolom
df_test.columns = df_test.columns.str.replace('^id-', 'id_', regex=True)    # replace id- to id_

#drop duplikasi data
duplicates = df.drop(columns=['TransactionID']).duplicated(keep=False)
df_duplicates = df[duplicates].drop(columns=['TransactionID'])
duplicated_index = df_duplicates.index[[1, 3, 5]]
df = df.drop(duplicated_index).reset_index(drop=True)

#drop kolom banyak missing values
drop_column = ['id_07','id_21', 'id_22', 'id_23','id_24', 'id_25', 'id_26', 'id_27', 'TransactionID']
df.drop(drop_column, axis=1, inplace=True)
df_test.drop(drop_column, axis=1, inplace=True)

#ubah missing values
categorical_cols = df.select_dtypes(include=['object', 'category']).columns
# print(categorical_cols)
for col in categorical_cols:
    df[col] = df[col].fillna('null')
for col in categorical_cols:
    df_test[col] = df_test[col].fillna('null')

for col in df.select_dtypes(include=np.number).columns:
    df[col] = df[col].fillna(-999)

for col in df_test.select_dtypes(include=np.number).columns:
    df_test[col] = df_test[col].fillna(-999)

#ubah ke numerik
encoders = {}
for col in categorical_cols:
    encoder = LabelEncoder()
    dff = pd.concat([df[col],df_test[col]], axis=0).sort_values().unique()
    encoder.fit(dff)
    # encoder.fit(pd.concat([train[col], test[col]], axis=0).unique())

    df[col] = encoder.transform(df[col])
    df_test[col] = encoder.transform(df_test[col])

    if df[col].max() > 32000 or df_test[col].max() > 32000:
        df[col] = df[col].astype('int32')
        df_test[col] = df_test[col].astype('int32')
    else:
        df[col] = df[col].astype('int16')
        df_test[col] = df_test[col].astype('int16')
    encoders[col] = encoder

#persiapan split data
x = df.drop('isFraud', axis=1)
y = df['isFraud']

#normalisasi data
scaler = StandardScaler()
x_scaled = scaler.fit_transform(x.astype(np.float32))
x_scaled = pd.DataFrame(x_scaled, columns=x.columns)
test_scaled = scaler.fit_transform(df_test.astype(np.float32))
test_scaled = pd.DataFrame(test_scaled, columns=df_test.columns)

#split data
x_train, x_val, y_train, y_val = train_test_split(x_scaled, y, test_size=0.2, random_state=42, stratify=y)

print("setelah preprocessing")
print("Train shape: ", x_train.shape)
print("Validation shape: ", x_val.shape)
print("Test shape: ", test_scaled.shape)

# modeling
# lr_model = LogisticRegression(random_state=42)

lgb_model = lgb.LGBMClassifier(
    learning_rate=0.01,
    max_depth=12,
    n_estimators=10000,
    bagging_fraction=0.8,
    feature_fraction=0.4,
    boosting_type= 'gbdt',
)

xgb_model = xgb.XGBClassifier(
    learning_rate=0.1,
    max_depth=9,
    n_estimators=200,
)

meta_model = MLPClassifier(hidden_layer_sizes=(50,))

stacked_model = StackingClassifier(
    classifiers=[xgb_model, lgb_model],
    meta_classifier=meta_model,
    use_probas=True,
    use_features_in_secondary=False
)

stacked_model.fit(x_train, y_train)

# predict
y_pred_prob = stacked_model.predict_proba(x_val)[:, 1]
y_pred = stacked_model.predict(x_val)
auc_roc_stack = roc_auc_score(y_val, y_pred_prob)

print(f"AUC-ROC score: {auc_roc_stack}")
print(classification_report(y_val, y_pred))
cm = confusion_matrix(y_val, y_pred)
print("Confusion Matrix:")
print(cm)

# simulasi 1 data
list_simulation = []

latency_times = []
len_test_scaled = len(test_scaled[:1000])
for i in range(len_test_scaled):
    start_time = time.time()
    result_fraud = stacked_model.predict(test_scaled.iloc[i:i+1])
    if result_fraud == 1:
        print(f"Fraud detected at index {i}")
    end_time = time.time()
    latency_times.append(end_time - start_time)

# Calculate and display latency statistics
average_latency = sum(latency_times) / len(latency_times)

list_simulation.append({
    'model': 'Stack',
    'batch_size': 1,
    'average_latency': average_latency,
    'total_latency': sum(latency_times),
    'max_latency': max(latency_times),
    'min_latency': min(latency_times)
})

# simulasi 100 data
batch_size = 100
num_batches = len(test_scaled) // batch_size
latency_times = []

for batch_idx in range(num_batches):
    start_time = time.time()
    batch_data = test_scaled.iloc[batch_idx * batch_size:(batch_idx + 1) * batch_size]
    result_fraud = stacked_model.predict(batch_data)  # Predict the batch
    end_time = time.time()

    # Calculate latency for the batch
    latency_times.append(end_time - start_time)
    print(f"Batch {batch_idx + 1}/{num_batches} processed. Latency: {latency_times[-1]:.6f} seconds")

# Calculate and display overall latency statistics
average_latency = sum(latency_times) / len(latency_times)
list_simulation.append({
    'model': 'Stack',
    'batch_size': batch_size,
    'average_latency': average_latency,
    'total_latency': sum(latency_times),
    'max_latency': max(latency_times),
    'min_latency': min(latency_times)
})

# simulasi 1000 data
batch_size = 1000
num_batches = len(test_scaled) // batch_size
latency_times = []

for batch_idx in range(num_batches):
    start_time = time.time()
    batch_data = test_scaled.iloc[batch_idx * batch_size:(batch_idx + 1) * batch_size]
    result_fraud = stacked_model.predict(batch_data)  # Predict the batch
    end_time = time.time()

    # Calculate latency for the batch
    latency_times.append(end_time - start_time)
    print(f"Batch {batch_idx + 1}/{num_batches} processed. Latency: {latency_times[-1]:.6f} seconds")

# Calculate and display overall latency statistics
average_latency = sum(latency_times) / len(latency_times)
list_simulation.append({
    'model': 'Stack',
    'batch_size': batch_size,
    'average_latency': average_latency,
    'total_latency': sum(latency_times),
    'max_latency': max(latency_times),
    'min_latency': min(latency_times)
})

df_simulation = pd.DataFrame(list_simulation)

Downloading...
From (original): https://drive.google.com/uc?id=1-LA_hivwi-LTmk3Il6zWc2SlWkw63lIp
From (redirected): https://drive.google.com/uc?id=1-LA_hivwi-LTmk3Il6zWc2SlWkw63lIp&confirm=t&uuid=db28b65b-f5ca-42b5-a599-36a0f2a874e1
To: /content/train_transaction.csv
100%|██████████| 683M/683M [00:07<00:00, 96.9MB/s]
Downloading...
From: https://drive.google.com/uc?id=1-3IcnQe1Vk6USpMFiZcc4plMM27u2BRM
To: /content/train_identity.csv
100%|██████████| 26.5M/26.5M [00:00<00:00, 48.8MB/s]
Downloading...
From (original): https://drive.google.com/uc?id=1--SQCKBo6r7JKFDJU1WLVSVzj7Wmblz1
From (redirected): https://drive.google.com/uc?id=1--SQCKBo6r7JKFDJU1WLVSVzj7Wmblz1&confirm=t&uuid=b8b747da-4c02-4aa8-b1f6-2a19713a58ee
To: /content/test_transaction.csv
100%|██████████| 613M/613M [00:05<00:00, 111MB/s]
Downloading...
From: https://drive.google.com/uc?id=1htmHneOewTUARR48VB5rF0k3tZIGpFcj
To: /content/test_identity.csv
100%|██████████| 25.8M/25.8M [00:00<00:00, 66.8MB/s]


data train:  (590540, 434)
data test:  (506691, 433)
